In [1]:
import pandas as pd
import glob

files = glob.glob("../data/*.csv")
print(files)

df = pd.read_csv(files[0])
print(df.shape)
print(df.columns.tolist())

[]


IndexError: list index out of range

In [2]:
files2 = glob.glob("../data/labelled_flows/*.csv")
print(files2)

df2 = pd.read_csv(files2[0])
print(df2.shape)
print(df2.columns.tolist())

['../data/labelled_flows\\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', '../data/labelled_flows\\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', '../data/labelled_flows\\Friday-WorkingHours-Morning.pcap_ISCX.csv', '../data/labelled_flows\\Monday-WorkingHours.pcap_ISCX.csv', '../data/labelled_flows\\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', '../data/labelled_flows\\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', '../data/labelled_flows\\Tuesday-WorkingHours.pcap_ISCX.csv', '../data/labelled_flows\\Wednesday-workingHours.pcap_ISCX.csv']
(225745, 85)
['Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Pack

In [3]:
# Clean column names (strip leading/trailing spaces)
df2.columns = df2.columns.str.strip()

# Check the label distribution
print(df2['Label'].value_counts())

# Check timestamp format
print(df2['Timestamp'].head())

# Check for missing values
print(df2.isnull().sum().sum(), "total missing values")

# Check for infinite values (common in Flow Bytes/s, Flow Packets/s)
import numpy as np
print((df2 == np.inf).sum().sum(), "total infinite values")

Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64
0    7/7/2017 3:30
1    7/7/2017 3:30
2    7/7/2017 3:30
3    7/7/2017 3:30
4    7/7/2017 3:30
Name: Timestamp, dtype: str
4 total missing values
64 total infinite values


In [4]:
# Drop rows with missing values (only 4, safe to drop)
df2 = df2.dropna()

# Replace infinite values with NaN, then drop those too
df2 = df2.replace([np.inf, -np.inf], np.nan).dropna()

# Confirm clean
print(df2.isnull().sum().sum(), "missing values after cleaning")
print(df2.shape)

# Parse timestamp properly
df2['Timestamp'] = pd.to_datetime(df2['Timestamp'], format='%d/%m/%Y %H:%M')
print(df2['Timestamp'].min(), "to", df2['Timestamp'].max())

0 missing values after cleaning
(225711, 85)
2017-07-07 03:30:00 to 2017-07-07 05:02:00


In [5]:
# Set time window size (5 minutes)
df2['time_window'] = df2['Timestamp'].dt.floor('5min')

# Group by source host + time window, aggregate behavioral stats
host_features = df2.groupby(['Source IP', 'time_window']).agg(
    connection_count=('Flow ID', 'count'),
    unique_destinations=('Destination IP', 'nunique'),
    unique_ports=('Destination Port', 'nunique'),
    total_fwd_packets=('Total Fwd Packets', 'sum'),
    total_bwd_packets=('Total Backward Packets', 'sum'),
    total_bytes_fwd=('Total Length of Fwd Packets', 'sum'),
    total_bytes_bwd=('Total Length of Bwd Packets', 'sum'),
    avg_flow_duration=('Flow Duration', 'mean'),
    attack_flow_ratio=('Label', lambda x: (x != 'BENIGN').mean())
).reset_index()

print(host_features.shape)
print(host_features.head(10))

(3819, 11)
         Source IP         time_window  connection_count  unique_destinations  \
0        1.1.70.73 2017-07-07 03:45:00                 1                    1   
1     1.193.219.24 2017-07-07 04:45:00                 2                    1   
2   101.69.185.208 2017-07-07 04:15:00                 6                    1   
3   101.69.185.240 2017-07-07 04:15:00                 1                    1   
4   101.69.185.240 2017-07-07 04:20:00                 1                    1   
5     103.43.91.16 2017-07-07 04:25:00                 3                    1   
6   104.105.77.229 2017-07-07 04:55:00                 2                    1   
7   104.105.77.229 2017-07-07 05:00:00                 1                    1   
8   104.106.241.83 2017-07-07 04:15:00                 2                    1   
9  104.106.249.194 2017-07-07 04:10:00                 4                    1   

   unique_ports  total_fwd_packets  total_bwd_packets  total_bytes_fwd  \
0             1        

In [8]:
host_features.to_csv("../data/host_features_friday.csv", index=False)
print("saved")

saved


In [9]:
suspicious = host_features[host_features['attack_flow_ratio'] > 0].sort_values(['Source IP', 'time_window'])
print(suspicious.head(20))

          Source IP         time_window  connection_count  \
763      172.16.0.1 2017-07-07 03:55:00             23865   
764      172.16.0.1 2017-07-07 04:00:00             33540   
765      172.16.0.1 2017-07-07 04:05:00             31051   
766      172.16.0.1 2017-07-07 04:10:00             31954   
767      172.16.0.1 2017-07-07 04:15:00              7622   
1613  192.168.10.50 2017-07-07 04:05:00              8717   

      unique_destinations  unique_ports  total_fwd_packets  total_bwd_packets  \
763                     1             1             103452              83704   
764                     1             1             150760             113222   
765                     1             1             145807             102773   
766                     1             1             142314             100789   
767                     6             2              30283              16327   
1613                   14          7250              33310              49531   

    